In [1]:
'''Pose landmark detection''' '''---TASK 2---'''

import cv2
import mediapipe as mp
import os

def draw_and_classify_pose(image_path, output_folder="Results_Pose"):
    mp_pose = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils # function inside the mediapipe library
    mp_drawing_styles = mp.solutions.drawing_styles

    image = cv2.imread(image_path)
    if image is None: return "no readings"

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    with mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5) as pose:
        # BGR -> RGB transformation and implementation (with openCV)
        results = pose.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        if not results.pose_landmarks:
            return "Cannot detect a body"

        # Drawing on the image
        annotated_image = image.copy()
        mp_drawing.draw_landmarks( #special function for mediapipe
            annotated_image,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style()
        )

        # Save the landmark image
        file_name = os.path.basename(image_path)
        cv2.imwrite(os.path.join(output_folder, f"pose_{file_name}"), annotated_image)

        # arm positions: 
        # Landmark 15: left wrist, 11: left shoulder
        # Landmark 16: right wrist, 12: right shoulder
        lm = results.pose_landmarks.landmark
        
        left_up = lm[15].y < lm[11].y
        right_up = lm[16].y < lm[12].y

        if left_up and right_up: return "Both arms up"
        elif left_up: return "Left arm up"
        elif right_up: return "Right arm up"
        else: return "Arms Down"

def process_pose_folder(folder_name):
    # check the folder path
    if not os.path.exists(folder_name):
        folder_name = os.path.join("..", folder_name)
    
    if not os.path.exists(folder_name):
        print(f"Error: '{folder_name}' cannot be found !")
        return

    print(f"{'File name':<25} | {'Position'}")
    print("-" * 50)

    for filename in os.listdir(folder_name):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            full_path = os.path.join(folder_name, filename)
            
            result = draw_and_classify_pose(full_path)
            print(f"{filename:<25} | {result}")

if __name__ == "__main__":
    target_folder = "TestData" 
    process_pose_folder(target_folder)

File name                 | Position
--------------------------------------------------
face-1.png                | Both arms up
face-2.png                | Right arm up
face-3.png                | Arms Down
pose-1.jpg                | Left arm up
pose-2.jpg                | Right arm up
pose-3.jpg                | Both arms up


In [3]:
'''Face landmark detection and Face direction detection together''' '''---TASK 3---'''

import cv2
import mediapipe as mp
import os


def draw_and_classify_face(image_path, output_folder="Results"):
    mp_face_mesh = mp.solutions.face_mesh
    mp_drawing = mp.solutions.drawing_utils # For drawing landmarks
    mp_drawing_styles = mp.solutions.drawing_styles # using drawing stayles inside the medipipe lib
    
    image = cv2.imread(image_path)
    if image is None: return "No readings"

    # Create an output folder (skip if exists)
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    with mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1) as face_mesh:
        results = face_mesh.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        if not results.multi_face_landmarks:
            return "Cannot detect face"

        # Drawing the landmarks
        annotated_image = image.copy()
        for face_landmarks in results.multi_face_landmarks:
            # 1. Draw the mesh(lines)
            mp_drawing.draw_landmarks(
                image=annotated_image,
                landmark_list=face_landmarks,
                connections=mp_face_mesh.FACEMESH_TESSELATION,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style()
            )
            # 2. draw eyes, lips and nose
            mp_drawing.draw_landmarks(
                image=annotated_image,
                landmark_list=face_landmarks,
                connections=mp_face_mesh.FACEMESH_CONTOURS,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style()
            )

        # Save the image
        file_name = os.path.basename(image_path)
        cv2.imwrite(os.path.join(output_folder, f"result_{file_name}"), annotated_image)

        # Direction detection
        nose = face_landmarks.landmark[1].x
        left_edge = face_landmarks.landmark[234].x
        right_edge = face_landmarks.landmark[454].x
        center_score = (nose - left_edge) / (right_edge - left_edge)

        # Center Score is our threshold to decide the direction of the face
        # if nose is close to the left side of the picture ---> LEFT side (nose - left_edge will be shorter) Center score will be low.
        if center_score < 0.4: return "Left"
        elif center_score > 0.6: return "Right"
        else: return "Straight"

#Creating the landmark images here by calling the functions
def process_folder(folder_name):
    if not os.path.exists(folder_name):
        folder_name = os.path.join("..", folder_name) 
    
    if not os.path.exists(folder_name):
        print(f"Error: '{folder_name}' Cannot be found!")
        return

    print(f"{'File Name':<25} | {'Direction'}")
    print("-" * 45)

    for filename in os.listdir(folder_name):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            full_path = os.path.join(folder_name, filename)
            
            # Find the full path of the images(FaceData Folder)
            result = draw_and_classify_face(full_path, output_folder="Results") #add the drawings into the Results folder
            print(f"{filename:<25} | {result}")

if __name__ == "__main__":
    target_folder = "TestData" 
    process_folder(target_folder) #---> CAll the function here

File Name                 | Direction
---------------------------------------------
face-1.png                | Left
face-2.png                | Right
face-3.png                | Straight
pose-1.jpg                | Cannot detect face
pose-2.jpg                | Straight
pose-3.jpg                | Cannot detect face


In [1]:
import cv2
import mediapipe as mp
import sys

def classify_arm_up(image_path):
    mp_pose = mp.solutions.pose
    # static_image_mode=True olması fotoğraflarda daha iyi sonuç verir
    with mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5) as pose:
        image = cv2.imread(image_path)
        if image is None: return "None"
        
        results = pose.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        if not results.pose_landmarks: return "None"

        lm = results.pose_landmarks.landmark
        
        # Koordinat sisteminde Y değeri yukarı gittikçe küçülür.
        # Sol bilek < Sol omuz ise sol kol yukarıdadır.
        left_up = lm[15].y < lm[11].y
        right_up = lm[16].y < lm[12].y

        if left_up and right_up: return "both"
        if left_up: return "left"
        if right_up: return "right"
        return "None"

if __name__ == "__main__":
    if len(sys.argv) > 1:
        print(classify_arm_up(sys.argv[1]))

None
